# Lecture 6 Tutorial: Attention 与 Llama 内部解析

对应 2026 第 6 讲：单头 Attention → MHA → GQA → 分块计算。

统一记号：B 为 batch size，S 为 sequence length，H 为 hidden size；多头时 H = h × head_dim，其中小写 h 表示头数。
各 batch 样本独立计算。本教学实现省略 RoPE、dropout、残差与 RMSNorm，聚焦 Attention 核心；不等同于完整的 Llama 层。

依次运行全部单元。检查 mask、因果性、batch 独立性、SDPA 数值对齐与分块等价性。


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)
print(f"PyTorch version: {torch.__version__}")


PyTorch version: 2.8.0


## 1. 从 hidden states 得到 Q/K/V

先使用普通单头/MHA 的等宽投影，Q、K、V 均为 [B,S,H]。GQA 将在第 4 节单独引入。
数学公式为 XW；PyTorch 的 `Linear.weight` 为 [out_features,in_features]，执行 X @ weight.T。
Llama 的 pre-norm Attention 接收归一化后的 hidden states，此处直接用随机张量作为输入。


In [2]:
batch_size, seq_len = 2, 4
num_heads = 4
head_dim = 8
hidden_size = num_heads * head_dim

hidden_states = torch.randn(batch_size, seq_len, hidden_size)

q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
k_proj_full = nn.Linear(hidden_size, hidden_size, bias=False)
v_proj_full = nn.Linear(hidden_size, hidden_size, bias=False)
o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

q = q_proj(hidden_states)
k = k_proj_full(hidden_states)
v = v_proj_full(hidden_states)

print(f"hidden_states shape: {hidden_states.shape}")
print(f"q shape: {q.shape}, k shape: {k.shape}, v shape: {v.shape}")


hidden_states shape: torch.Size([2, 4, 32])
q shape: torch.Size([2, 4, 32]), k shape: torch.Size([2, 4, 32]), v shape: torch.Size([2, 4, 32])


## 2. 单头注意力 (Scaled Dot-Product)

下面按照讲义的推导，逐步计算 $P$、$\text{softmax}$ 与输出 $O$。为简洁起见，这里只展示因果 mask（decoder 的自回归约束）。


In [3]:
scale = 1.0 / math.sqrt(hidden_size)
scores = torch.matmul(q, k.transpose(-2, -1)) * scale
causal_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
scores_masked = scores.masked_fill(causal_mask, float('-inf'))
attn_weights = F.softmax(scores_masked, dim=-1)
context_single = torch.matmul(attn_weights, v)

print('scores shape:', scores.shape)
print('attention weights sums:', attn_weights.sum(dim=-1))
print('context_single shape:', context_single.shape)
print(causal_mask)
print(scores_masked)

scores shape: torch.Size([2, 4, 4])
attention weights sums: tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000, 1.0000]], grad_fn=<SumBackward1>)
context_single shape: torch.Size([2, 4, 32])
tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])
tensor([[[ 0.1864,    -inf,    -inf,    -inf],
         [-0.2687, -0.1780,    -inf,    -inf],
         [ 0.2653, -0.3983,  0.2157,    -inf],
         [-0.1443, -0.0368,  0.2042,  0.0041]],

        [[ 0.0577,    -inf,    -inf,    -inf],
         [ 0.1112,  0.0621,    -inf,    -inf],
         [ 0.0612,  0.0685, -0.3430,    -inf],
         [-0.5373, -0.0201, -0.4858,  0.3036]]], grad_fn=<MaskedFillBackward0>)


In [4]:
def stable_softmax(x: torch.Tensor) -> torch.Tensor:
    # 教学前提：每个 Query 行至少存在一个未屏蔽且有限的 score。
    if not torch.isfinite(x).any(dim=-1).all():
        raise ValueError("Each query row must have a finite, unmasked score")
    x_max = x.max(dim=-1, keepdim=True).values
    x_exp = torch.exp(x - x_max)
    return x_exp / x_exp.sum(dim=-1, keepdim=True)

manual_weights = stable_softmax(scores_masked)
torch.testing.assert_close(manual_weights, attn_weights)
torch.testing.assert_close(attn_weights.sum(dim=-1), torch.ones(batch_size, seq_len))
assert torch.count_nonzero(attn_weights.masked_select(causal_mask.unsqueeze(0))) == 0
print('手写 softmax、行和与未来位置零权重检查通过')
print('第一个样本的注意力权重：\n', attn_weights[0])


手写 softmax、行和与未来位置零权重检查通过
第一个样本的注意力权重：
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4774, 0.5226, 0.0000, 0.0000],
        [0.4054, 0.2088, 0.3858, 0.0000],
        [0.2132, 0.2374, 0.3021, 0.2473]], grad_fn=<SelectBackward0>)


### 因果性与 batch 独立性实验

只改变最后一个 token，前面位置的输出应保持不变。然后改变另一个 batch 样本，当前样本输出也不应改变。
这里函数的 bool mask 约定为 True 表示屏蔽，后续 SDPA 调用直接使用 is_causal=True，避免混淆不同 API 的 mask 语义。

In [5]:
def single_attention_from_hidden(x, causal=True):
    q_x, k_x, v_x = q_proj(x), k_proj_full(x), v_proj_full(x)
    logits = q_x @ k_x.transpose(-2, -1) / math.sqrt(x.size(-1))
    if causal:
        logits = logits.masked_fill(causal_mask, float('-inf'))
    return F.softmax(logits, dim=-1) @ v_x

changed = hidden_states.clone()
changed[:, -1, :] += torch.linspace(-5, 5, hidden_size)
baseline = single_attention_from_hidden(hidden_states)
causal_changed = single_attention_from_hidden(changed)
torch.testing.assert_close(baseline[:, :-1], causal_changed[:, :-1])
print('修改最后一个 token 后，前面位置的 causal 输出保持不变')
no_mask_diff = (single_attention_from_hidden(hidden_states, False)[:, :-1]
                - single_attention_from_hidden(changed, False)[:, :-1]).abs().max()
print('去掉 causal mask，前面位置的最大变化:', no_mask_diff.item())
assert no_mask_diff.item() > 1e-5

other_batch = hidden_states.clone()
other_batch[1] += 10
torch.testing.assert_close(baseline[0], single_attention_from_hidden(other_batch)[0])
print('改变另一个 batch 样本，当前样本输出保持不变')

修改最后一个 token 后，前面位置的 causal 输出保持不变
去掉 causal mask，前面位置的最大变化: 1.8575148582458496
改变另一个 batch 样本，当前样本输出保持不变


### Padding 与 causal mask 的组合

示例使用右侧 padding：第二个样本只有前两个位置有效。屏蔽 Key 列后，仅对有效 Query 输出作检查；padding Query 的输出应在后续使用或 loss 中忽略。
整行均被屏蔽时普通 softmax 会产生 NaN，因此不能将这样的行当作有效 Query。

In [6]:
valid_tokens = torch.tensor([[True, True, True, True], [True, True, False, False]])
combined_mask = causal_mask.unsqueeze(0) | (~valid_tokens).unsqueeze(1)
padded_weights = stable_softmax(scores.masked_fill(combined_mask, float('-inf')))
assert torch.count_nonzero(padded_weights[1, :2, 2:]) == 0
torch.testing.assert_close(padded_weights.sum(-1)[valid_tokens],
                           torch.ones_like(padded_weights.sum(-1)[valid_tokens]))
print('第二个样本的有效 Query 权重：\n', padded_weights[1, :2])

第二个样本的有效 Query 权重：
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5123, 0.4877, 0.0000, 0.0000]], grad_fn=<SliceBackward0>)


## 3. Multi-Head Attention 的张量变换

把 $Q/K/V$ 拆成多个头可以让模型在不同的子空间里捕捉特征。注意 reshape、transpose 的顺序与 `Concat` 聚合回原维度的细节。


In [7]:
q_heads = q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
k_heads = k.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
v_heads = v.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)

print('q_heads shape:', q_heads.shape)

causal_mask_h = causal_mask.unsqueeze(0).unsqueeze(0)
scores_multi = torch.matmul(q_heads, k_heads.transpose(-2, -1)) / math.sqrt(head_dim)
scores_multi = scores_multi.masked_fill(causal_mask_h, float('-inf'))
weights_multi = F.softmax(scores_multi, dim=-1)
context_multi = torch.matmul(weights_multi, v_heads)
context_multi = context_multi.transpose(1, 2).contiguous().view(batch_size, seq_len, hidden_size)
output_multi = o_proj(context_multi)

print('context_multi shape:', context_multi.shape)
print('output_multi shape:', output_multi.shape)

# SDPA 使用同一 Q/K/V、causal mask 与缩放规则作参考。
reference_heads = F.scaled_dot_product_attention(q_heads, k_heads, v_heads,
                                                dropout_p=0.0, is_causal=True)
reference = o_proj(reference_heads.transpose(1, 2).contiguous().view(batch_size, seq_len, hidden_size))
torch.testing.assert_close(output_multi, reference, atol=1e-6, rtol=1e-5)
print('MHA 与 PyTorch SDPA 对齐通过')


q_heads shape: torch.Size([2, 4, 4, 8])
context_multi shape: torch.Size([2, 4, 32])
output_multi shape: torch.Size([2, 4, 32])
MHA 与 PyTorch SDPA 对齐通过


## 4. Grouped Query Attention (GQA)

将 h 个 Query 头分成 g 组，每组 h/g 个 Query 头共享一个 K 头和一个 V 头。
Q 投影宽度为 h × head_dim，K/V 各为 g × head_dim。具体模型是否采用 GQA 应查看配置。

这里用 `repeat_interleave` 显式复制 KV 来演示共享关系，实际缓存只保存未扩展的 g 个 KV 头。该演示不代表实际内核的显存或速度表现。


In [8]:
num_key_value_heads = 2
kv_proj_dim = num_key_value_heads * head_dim

k_proj_gqa = nn.Linear(hidden_size, kv_proj_dim, bias=False)
v_proj_gqa = nn.Linear(hidden_size, kv_proj_dim, bias=False)

k_gqa_raw = k_proj_gqa(hidden_states)
v_gqa_raw = v_proj_gqa(hidden_states)

k_kv = k_gqa_raw.view(batch_size, seq_len, num_key_value_heads, head_dim).transpose(1, 2).contiguous()
v_kv = v_gqa_raw.view(batch_size, seq_len, num_key_value_heads, head_dim).transpose(1, 2).contiguous()


def repeat_kv(hidden_states: torch.Tensor, target_heads: int) -> torch.Tensor:
    if target_heads % hidden_states.size(1) != 0:
        raise ValueError("Query head count must be divisible by KV head count")
    repeat_factor = target_heads // hidden_states.size(1)
    return hidden_states.repeat_interleave(repeat_factor, dim=1)


k_gqa_expanded = repeat_kv(k_kv, num_heads)
v_gqa_expanded = repeat_kv(v_kv, num_heads)

scores_gqa = torch.matmul(q_heads, k_gqa_expanded.transpose(-2, -1)) / math.sqrt(head_dim)
scores_gqa = scores_gqa.masked_fill(causal_mask_h, float('-inf'))
weights_gqa = F.softmax(scores_gqa, dim=-1)
context_gqa = torch.matmul(weights_gqa, v_gqa_expanded)
context_gqa = context_gqa.transpose(1, 2).contiguous().view(batch_size, seq_len, hidden_size)

print('k_heads (full) shape:', k_heads.shape)
print('k_kv (GQA cached) shape:', k_kv.shape)
full_kv_elements = k_heads.numel() + v_heads.numel()
gqa_kv_elements = k_kv.numel() + v_kv.numel()
print(f'Full K+V cache elements: {full_kv_elements:,}')
print(f'GQA K+V cache elements: {gqa_kv_elements:,}')
assert gqa_kv_elements * num_heads == full_kv_elements * num_key_value_heads
print(f'Cache ratio (GQA/MHA): {gqa_kv_elements / full_kv_elements:.2f}')
print('context_gqa shape:', context_gqa.shape)

output_gqa = o_proj(context_gqa)
assert output_gqa.shape == hidden_states.shape
print("GQA output shape:", output_gqa.shape)


k_heads (full) shape: torch.Size([2, 4, 4, 8])
k_kv (GQA cached) shape: torch.Size([2, 2, 4, 8])
Full K+V cache elements: 512
GQA K+V cache elements: 256
Cache ratio (GQA/MHA): 0.50
context_gqa shape: torch.Size([2, 4, 32])
GQA output shape: torch.Size([2, 4, 32])


## 5. 分块 Attention：验证等价性

对每个 Query 块计算各个 Key 块的 score，拼接成完整的 score 行，再做 softmax 与 Value 聚合。
这里没有实现 Online Softmax 或 FlashAttention。每个 Query 块仍保存 [B,block_size,S] 的 score。
Python 循环不代表 GPU 加速；训练时 autograd 可能保留各块中间量，不能据此推断训练显存收益。


In [9]:
def scaled_dot_product_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
    dim = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(dim)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, v)


def blocked_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, block_size: int, mask: torch.Tensor | None = None) -> torch.Tensor:
    if block_size <= 0:
        raise ValueError("block_size must be positive")
    bs, seq_len, dim = q.shape
    outputs = torch.zeros_like(q)

    for start in range(0, seq_len, block_size):
        end = min(start + block_size, seq_len)
        q_block = q[:, start:end, :]

        scores_blocks = []
        value_blocks = []
        for k_start in range(0, seq_len, block_size):
            k_end = min(k_start + block_size, seq_len)
            k_block = k[:, k_start:k_end, :]
            scores = torch.matmul(q_block, k_block.transpose(-2, -1)) / math.sqrt(dim)
            if mask is not None:
                mask_block = mask[:, start:end, k_start:k_end]
                scores = scores.masked_fill(mask_block, float('-inf'))
            scores_blocks.append(scores)
            value_blocks.append(v[:, k_start:k_end, :])

        scores_concat = torch.cat(scores_blocks, dim=-1)
        weights = F.softmax(scores_concat, dim=-1)
        values_concat = torch.cat(value_blocks, dim=1)
        output_block = torch.matmul(weights, values_concat)
        outputs[:, start:end, :] = output_block

    return outputs


In [10]:
mask = causal_mask.unsqueeze(0).expand(batch_size, -1, -1)
full_attention = scaled_dot_product_attention(q, k, v, mask)
# 包括不能整除序列长度的块，以及大于序列长度的块。
for block_size in (1, 2, 3, 8):
    blocked_out = blocked_attention(q, k, v, block_size=block_size, mask=mask)
    torch.testing.assert_close(blocked_out, full_attention, atol=1e-6, rtol=1e-5)
    print(f'block_size={block_size}, max diff={(full_attention - blocked_out).abs().max().item():.3g}')

flat_scores = torch.zeros(4)
whole = F.softmax(flat_scores, dim=-1)
separate = torch.cat([F.softmax(block, dim=-1) for block in flat_scores.split(2)])
print('整行 softmax:', whole)
print('每块单独 softmax 后拼接（错误）:', separate)
assert not torch.allclose(whole, separate)


block_size=1, max diff=5.96e-08
block_size=2, max diff=5.96e-08
block_size=3, max diff=5.96e-08
block_size=8, max diff=0
整行 softmax: tensor([0.2500, 0.2500, 0.2500, 0.2500])
每块单独 softmax 后拼接（错误）: tensor([0.5000, 0.5000, 0.5000, 0.5000])


## 6. 课堂回顾与下一讲

- 因果性：改变最后一个 token，前面位置的输出不变；不同 batch 样本之间也互不影响。
- MHA：Q/K/V 从 [B,S,H] 变成 [B,h,S,head_dim]，softmax 沿 Key 维度执行。
- GQA：缓存统计必须包含 K 与 V；缓存保留 g 个头，输出仍是 [B,S,H]。
- 分块：softmax 分母依赖所有可访问的 Key，不能各块单独归一化后直接拼接。
- 下一讲：能否不拼接完整 score 行，增量维护归一化因子和加权和？最大值改变时如何修正旧的累积量？
